In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ouro")
spark.sql("SELECT * FROM prata.eventos_presenca").createOrReplaceTempView("prata_presenca_view")
spark.sql("SELECT * FROM prata.eventos").createOrReplaceTempView("prata_eventos_view")
print("Views criadas.")

# Entrega 1: Score de Engajamento

## O que mede
Um número de 0 a 100 que resume o nível de participação do deputado nas atividades legislativas.

## Componentes do score
O score é composto por 3 fatores, cada um com um peso:

1. **Total de eventos (peso 40%)**
   Quantos eventos distintos o deputado participou no ano.
   Ex: 194 eventos = 194 × 0.4 = 77,6 pontos

2. **Meses ativos (peso 30%)**
   Em quantos meses diferentes o deputado teve pelo menos um evento.
   Ex: 11 meses = 11 × 2.0 × 0.3 = 6,6 pontos
   O fator 2.0 serve para equiparar a escala com os outros componentes.

3. **Tipos de evento (peso 30%)**
   Quantos tipos diferentes de evento o deputado participou.
   Ex: 13 tipos = 13 × 3.0 × 0.3 = 11,7 pontos
   O fator 3.0 serve para equiparar a escala.

## Fórmula final
score = (total_eventos × 0.4) + (meses_ativos × 2.0 × 0.3) + (tipos_evento × 3.0 × 0.3)

## Exemplo real
Dr. Luiz Ovando (PP-MS):
- 194 eventos × 0.4 = 77,6
- 11 meses × 2.0 × 0.3 = 6,6
- 13 tipos × 3.0 × 0.3 = 11,7
- Score total = 95,9

## Limitação
O score usa apenas dados de presença em eventos. Votações, discursos e requerimentos
não estão incluídos (API indisponível).

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.score_engajamento
USING DELTA
COMMENT 'Camada Ouro - Score de engajamento parlamentar (2024)'
AS
WITH eventos_por_mes AS (
    SELECT 
        id_deputado,
        mes,
        COUNT(DISTINCT id_evento) AS eventos_mes
    FROM prata_presenca_view
    WHERE ano = 2024
    GROUP BY id_deputado, mes
),
regularidade_dep AS (
    SELECT 
        id_deputado,
        ROUND(STDDEV(eventos_mes), 1) AS regularidade
    FROM eventos_por_mes
    GROUP BY id_deputado
),
engajamento AS (
    SELECT 
        p.id_deputado,
        MAX(p.nome_deputado) AS nome_deputado,
        MAX(p.sigla_partido) AS sigla_partido,
        MAX(p.sigla_uf) AS sigla_uf,
        COUNT(DISTINCT p.id_evento) AS total_eventos,
        COUNT(DISTINCT p.mes) AS meses_ativos,
        COUNT(DISTINCT p.descricao_tipo) AS tipos_evento,
        AVG(COUNT(DISTINCT p.id_evento)) OVER () AS media_geral_eventos,
        COALESCE(MAX(r.regularidade), 0) AS regularidade
    FROM prata_presenca_view p
    LEFT JOIN regularidade_dep r ON p.id_deputado = r.id_deputado
    WHERE p.ano = 2024
    GROUP BY p.id_deputado
)
SELECT 
    *,
    ROUND(total_eventos * 0.4 + meses_ativos * 2.0 * 0.3 + tipos_evento * 3.0 * 0.3, 1) AS score_engajamento
FROM engajamento
WHERE meses_ativos >= 3
ORDER BY score_engajamento DESC

In [0]:
%sql
-- Top 10 deputados mais engajados
SELECT 
    nome_deputado,
    sigla_partido,
    sigla_uf,
    total_eventos,
    meses_ativos,
    tipos_evento,
    score_engajamento
FROM ouro.score_engajamento
ORDER BY score_engajamento DESC
LIMIT 10

In [0]:
%sql
-- Top 10 deputados MENOS engajados
SELECT 
    nome_deputado,
    sigla_partido,
    sigla_uf,
    total_eventos,
    meses_ativos,
    tipos_evento,
    score_engajamento
FROM ouro.score_engajamento
ORDER BY score_engajamento ASC
LIMIT 10

# Entrega 2: Detecção de Padrão de Ausência

## O que mede
Identifica deputados que participam de poucos eventos em comparação com o total
de eventos disponíveis de cada tipo.

## Como funciona
Para cada tipo de evento (ex: Sessão Deliberativa), calcula-se:
- Total de eventos daquele tipo no ano (ex: 93 Sessões Deliberativas em 2024)
- Quantos eventos cada deputado participou
- Taxa de presença = eventos_participados / total_eventos * 100

## Exemplo
- Sessão Deliberativa teve 93 eventos em 2024
- Deputado X participou de apenas 1
- Taxa de presença = 1/93 * 100 = 1,1%
- Ausências = 92 eventos perdidos

## Interpretação
- Taxa abaixo de 10%: absenteísmo grave
- Taxa entre 10% e 50%: participação abaixo da média
- Taxa acima de 80%: alta participação

## Limitação
O dado reflete presença registrada em eventos, não necessariamente ausência
injustificada. Deputados podem ter faltas justificadas por licença médica,
missões oficiais, etc.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.padrao_ausencia
USING DELTA
COMMENT 'Camada Ouro - Padrao de ausencia por tipo de evento'
AS
WITH total_eventos_tipo AS (
    SELECT 
        descricao_tipo,
        COUNT(DISTINCT id_evento) AS total_eventos_tipo
    FROM prata_eventos_view
    WHERE ano = 2024
    GROUP BY descricao_tipo
),
presenca_deputado_tipo AS (
    SELECT 
        p.id_deputado,
        MAX(p.nome_deputado) AS nome_deputado,
        MAX(p.sigla_partido) AS sigla_partido,
        MAX(p.sigla_uf) AS sigla_uf,
        p.descricao_tipo,
        COUNT(DISTINCT p.id_evento) AS eventos_participados
    FROM prata_presenca_view p
    WHERE p.ano = 2024
    GROUP BY p.id_deputado, p.descricao_tipo
)
SELECT 
    pt.id_deputado,
    pt.nome_deputado,
    pt.sigla_partido,
    pt.sigla_uf,
    pt.descricao_tipo,
    pt.eventos_participados,
    t.total_eventos_tipo,
    ROUND(pt.eventos_participados * 100.0 / t.total_eventos_tipo, 1) AS taxa_presenca_pct,
    t.total_eventos_tipo - pt.eventos_participados AS ausencias
FROM presenca_deputado_tipo pt
JOIN total_eventos_tipo t ON pt.descricao_tipo = t.descricao_tipo
ORDER BY taxa_presenca_pct ASC

In [0]:
%sql
-- Deputados com mais ausencias em Sessoes Deliberativas
SELECT 
    nome_deputado,
    sigla_partido,
    sigla_uf,
    descricao_tipo,
    eventos_participados,
    total_eventos_tipo,
    taxa_presenca_pct,
    ausencias
FROM ouro.padrao_ausencia
WHERE descricao_tipo = 'Sessão Deliberativa'
ORDER BY taxa_presenca_pct ASC
LIMIT 10

# Entrega 3: Série Temporal de Engajamento
Evolução mensal da presença média dos deputados ao longo de 2024.

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.serie_temporal_engajamento
USING DELTA
COMMENT 'Camada Ouro - Serie temporal de engajamento mensal (2024)'
AS
SELECT 
    mes,
    COUNT(DISTINCT id_deputado) AS deputados_ativos,
    COUNT(DISTINCT id_evento) AS total_eventos,
    ROUND(COUNT(DISTINCT id_evento) * 1.0 / COUNT(DISTINCT id_deputado), 1) AS eventos_por_deputado
FROM prata_presenca_view
WHERE ano = 2024
GROUP BY mes
ORDER BY mes

In [0]:
%sql
SELECT 
    mes,
    deputados_ativos,
    total_eventos,
    eventos_por_deputado,
    CASE 
        WHEN mes IN (7, 8, 9, 10) THEN 'Período eleitoral'
        ELSE 'Normal'
    END AS periodo
FROM ouro.serie_temporal_engajamento

# Explicação: Media Geral, Diferença da Media e Percentil

## Media Geral (69,6)
É a média do score de engajamento de TODOS os deputados.
Se todos tivessem o mesmo nível de participação, todos teriam score 69,6.

## Diferença da Média
Indica quantos pontos o deputado está ACIMA ou ABAIXO da média geral.

Exemplos:
- Dr. Luiz Ovando: diferença de +26,3 → está 26,3 pontos ACIMA da média (excelente)
- Um deputado com score 50: diferença de -19,6 → está 19,6 pontos ABAIXO da média (ruim)

Valores positivos = acima da média / Valores negativos = abaixo da média.

## Percentil
Indica a posição do deputado em relação a todos os outros, de 0 a 100.

- Percentil 100: o deputado MAIS engajado da Câmara (top 1)
- Percentil 99,8: Daniela Reinehr está entre os 0,2% mais engajados
- Percentil 50: o deputado está exatamente na mediana (metade acima, metade abaixo)
- Percentil 0: o deputado MENOS engajado

## Interpretação do ranking atual
- Dr. Luiz Ovando: percentil 100 (o mais engajado), 26,3 pontos acima da média
- Todos os top 10 estão entre os 2% mais engajados (percentil acima de 98)
- A média geral é 69,6, o que significa que o deputado típico tem esse score

In [0]:
%sql
CREATE OR REPLACE TABLE ouro.relatorio_mensal_deputado
USING DELTA
COMMENT 'Camada Ouro - Relatorio mensal com percentil por deputado'
AS
WITH stats AS (
    SELECT 
        AVG(score_engajamento) AS media_geral,
        STDDEV(score_engajamento) AS desvio
    FROM ouro.score_engajamento
)
SELECT 
    s.id_deputado,
    s.nome_deputado,
    s.sigla_partido,
    s.sigla_uf,
    s.total_eventos,
    s.meses_ativos,
    s.tipos_evento,
    s.score_engajamento,
    ROUND(stats.media_geral, 1) AS media_geral,
    ROUND(s.score_engajamento - stats.media_geral, 1) AS diferenca_da_media,
    ROUND(PERCENT_RANK() OVER (ORDER BY s.score_engajamento) * 100, 1) AS percentil
FROM ouro.score_engajamento s
CROSS JOIN stats
ORDER BY percentil DESC

In [0]:
%sql
SELECT 
    nome_deputado,
    sigla_partido,
    sigla_uf,
    score_engajamento,
    media_geral,
    diferenca_da_media,
    percentil
FROM ouro.relatorio_mensal_deputado
ORDER BY percentil DESC
LIMIT 10
